# Mean-difference random-label selectivity

This Colab notebook runs **mean_diff** on ToyMovieReview and full AIT using
last-non-padding-token residual activations. It has separate GPT-2 Small and
Qwen3-0.6B Base sections, but both write beneath one shared Google Drive run ID.

Mean difference has no hyperparameter search. Its learned centroid direction is evaluated with the training-class midpoint decision rule.

Every random-label seed is paired with a real-task run using the same optimization
seed. Labels are permuted independently within train, validation, and test while
preserving each split's class counts. Hyperparameters are selected on the real
validation split only; test data never affects selection. Result tables are written
only after both the real and paired random-label fits finish. Combined plots are
created only when all four methods and both models are present.

## 1. Settings

Keep the same `RUN_ID` in all four notebooks. Change it before starting a new study.
The configured fixed residual boundaries are:

| Dataset | GPT-2 Small | Qwen3-0.6B Base |
|---|---:|---:|
| ToyMovieReview | 10 | 26 |
| Full AIT | 11 | 26 |

In [1]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional immutable commit or tag.
DRIVE_ROOT = "/content/drive/MyDrive/sentiment-geometry/selectivity"
RUN_ID = "fixed-layer-selectivity-v1"  # Use this exact value in all four notebooks.

METHOD = "mean_diff"
DEVICE = "cuda"
DTYPE = "auto"
RUN_EXPERIMENT = True
SHOW_PROGRESS = True
HF_TOKEN_SOURCE = "prompt"  # "prompt" (hidden entry) or "colab_secret".

MANUAL_TRIALS = []  # Mean difference has no tunable hyperparameters.

## 2. Install the package, verify imports, and mount Drive

The notebook imports the editable checkout, verifies its filesystem location, imports
every package module, and checks the public APIs used below. This catches stale Colab
modules before a long run starts. The notebook calls package APIs rather than
reimplementing extraction, fitting, randomization, evaluation, or persistence.

In [2]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
if PROJECT_REVISION is not None:
    subprocess.run(
        ["git", "checkout", "--detach", PROJECT_REVISION],
        cwd=PROJECT_ROOT,
        check=True,
    )
project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
root_string = str(PROJECT_ROOT.resolve())
if root_string in sys.path:
    sys.path.remove(root_string)
sys.path.insert(0, root_string)
for module_name in list(sys.modules):
    if module_name == "sentiment_geometry" or module_name.startswith("sentiment_geometry."):
        del sys.modules[module_name]
importlib.invalidate_caches()

sentiment_geometry = importlib.import_module("sentiment_geometry")
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )
module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(
        sentiment_geometry.__path__, prefix="sentiment_geometry."
    )
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)
required_apis = {
    "sentiment_geometry.experiments.selectivity": {
        "FixedLayerSelectivityConfig", "run_fixed_layer_selectivity"
    },
    "sentiment_geometry.reporting": {
        "combine_fixed_layer_selectivity_runs",
        "load_fixed_layer_selectivity_report",
        "plot_fixed_layer_selectivity",
        "plot_fixed_layer_run_diagnostics",
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(f"{module_name} is missing {missing_apis}.")

drive.mount("/content/drive")
RUN_ROOT = Path(DRIVE_ROOT) / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)
print(f"Imported and checked {len(module_names)} package modules.")
print("Shared run root:", RUN_ROOT)

Mounted at /content/drive
Project commit: 4d63263fc02c8332a2fcacf5d4fa5205d62b170c
Imported package from: /content/sentiment-manifold/sentiment_geometry
Imported and checked 80 package modules.
Shared run root: /content/drive/MyDrive/sentiment-geometry/selectivity/fixed-layer-selectivity-v1


## 3. Authenticate to Hugging Face

The default `HF_TOKEN_SOURCE = "prompt"` asks for the token through a hidden manual
prompt, matching the earlier notebooks. To use Colab Secrets instead, add a secret
named `HF_TOKEN` and set `HF_TOKEN_SOURCE = "colab_secret"`. The token is kept only in
runtime memory, is never printed or written to Drive, and is placed in the environment
only while a model run is actively loading AIT data.

In [3]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name in _RUNTIME_SECRETS:
        return _RUNTIME_SECRETS[name]
    if HF_TOKEN_SOURCE == "prompt":
        value = getpass(f"Enter {name} (input hidden): ").strip()
    elif HF_TOKEN_SOURCE == "colab_secret":
        from google.colab import userdata

        value = (userdata.get(name) or "").strip()
    else:
        raise ValueError("HF_TOKEN_SOURCE must be 'prompt' or 'colab_secret'.")
    if not value:
        raise RuntimeError(f"{name} was not provided.")
    _RUNTIME_SECRETS[name] = value
    return value

def release_hf_environment(env_name="HF_TOKEN"):
    os.environ.pop(env_name, None)
    reset_sessions()
    gc.collect()

def clear_hf_credentials(env_name="HF_TOKEN"):
    release_hf_environment(env_name)
    value = _RUNTIME_SECRETS.pop("HF_TOKEN", None)
    if value is not None:
        del value

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
except BaseException:
    clear_hf_credentials()
    raise
finally:
    del _token
print(
    f"Authenticated to Hugging Face as {hf_account}. "
    "Token value was not displayed."
)

Authenticated to Hugging Face as kokolamba. Token value was not displayed.


## 4. Verify the protocol and define the reusable runner

A trial list contains complete hand-chosen configurations, not a grid. There are no
more than three user-adjusted hyperparameters for this method. Each model run includes
both datasets and all paired real/random-label seeds before any result table is shown.

In [4]:
import pandas as pd
from IPython.display import Image, display

from sentiment_geometry.experiments.selectivity import (
    FixedLayerSelectivityConfig,
    run_fixed_layer_selectivity,
)
from sentiment_geometry.reporting import (
    combine_fixed_layer_selectivity_runs,
    load_fixed_layer_selectivity_report,
    plot_fixed_layer_selectivity,
    plot_fixed_layer_run_diagnostics,
)

CONFIG_PATH = PROJECT_ROOT / "configs/selectivity/fixed_layer.yaml"
EXPECTED_LAYERS = {
    "gpt2-small": {"toy_movie_review": 10, "full_ait": 11},
    "qwen-0.6b": {"toy_movie_review": 26, "full_ait": 26},
}

def display_table(title, frame, columns=None):
    print(title)
    selected = frame
    if columns is not None:
        selected = frame[[column for column in columns if column in frame.columns]]
    display(selected.round(4))

def configured_run(model_name):
    config = FixedLayerSelectivityConfig.load(CONFIG_PATH)
    config.models = [model for model in config.models if model.name == model_name]
    if len(config.models) != 1:
        raise ValueError(f"Expected exactly one config for {model_name}")
    model = config.models[0]
    model.device = DEVICE
    model.dtype = DTYPE
    actual_layers = {
        dataset: model.layer_for(dataset) for dataset in config.data.datasets
    }
    if actual_layers != EXPECTED_LAYERS[model_name]:
        raise ValueError(
            f"Layer contract changed for {model_name}: {actual_layers}"
        )
    config.methods = [METHOD]
    config.output.output_dir = str(RUN_ROOT / "methods" / METHOD)
    config.output.run_id = model_name
    config.progress.enabled = SHOW_PROGRESS
    pass
    config.validate(require_layers=True)
    return config

def run_and_display(model_name):
    config = configured_run(model_name)
    run_dir = Path(config.output.output_dir) / config.output.run_id
    if RUN_EXPERIMENT:
        token_env = config.data.hf_token_env
        os.environ[token_env] = get_runtime_secret("HF_TOKEN")
        try:
            run_dir = run_fixed_layer_selectivity(config)
        finally:
            release_hf_environment(token_env)
    if not (run_dir / "metrics.csv").is_file():
        raise FileNotFoundError(f"No completed results at {run_dir}")

    report = load_fixed_layer_selectivity_report(run_dir)
    if not report.tuning_trials.empty:
        display_table(
            "Real-validation tuning trials (selected rows are marked):",
            report.tuning_trials,
            [
                "dataset", "layer", "trial_index", "hyperparameters",
                "validation_native_accuracy",
                "validation_native_balanced_accuracy",
                "validation_midpoint_balanced_accuracy",
                "validation_loss", "validation_recovery",
                "validation_logit_flip", "validation_sign_flip", "selected",
            ],
        )

    metric_columns = [
        "model", "dataset", "layer", "task", "split", "n_seeds",
        "native_balanced_accuracy_mean", "native_balanced_accuracy_std",
        "midpoint_balanced_accuracy_mean", "midpoint_balanced_accuracy_std",
        "prediction_agreement_mean",
    ]
    display_table(
        "Training memorization diagnostic (real and random labels):",
        report.training_memorization,
        metric_columns,
    )
    display_table(
        "All-split performance, mean and standard deviation across paired seeds:",
        report.performance,
        metric_columns,
    )
    display_table(
        "Native versus midpoint gaps and prediction agreement:",
        report.native_midpoint_comparison,
        [
            "model", "dataset", "layer", "task", "split", "n_seeds",
            "accuracy_gap_midpoint_minus_native_mean",
            "balanced_gap_midpoint_minus_native_mean",
            "prediction_agreement_mean",
        ],
    )
    balanced_selectivity = report.selectivity[
        report.selectivity["metric"].isin(
            ["native_balanced_accuracy", "midpoint_balanced_accuracy"]
        )
    ]
    display_table(
        "Paired selectivity, including train, validation, and test:",
        balanced_selectivity,
        [
            "model", "dataset", "layer", "split", "metric",
            "selectivity_mean", "selectivity_std", "selectivity_ci95_low",
            "selectivity_ci95_high", "n_seeds",
        ],
    )
    display_table(
        "Fit diagnostics and selected training epochs:",
        report.fit_diagnostics,
        [
            "model", "dataset", "layer", "task", "seed", "hyperparameters",
            "n_parameters", "best_epoch", "epochs_completed_from_history",
            "best_validation_loss", "best_training_loss",
        ],
    )
    if not report.das_causal_metrics.empty:
        display_table(
            "DAS IIA, patched/clean/corrupted accuracy, recovery, "
            "logit flip, and sign flip:",
            report.das_causal_metrics,
            [
                "model", "dataset", "layer", "task", "split", "n_seeds",
                "iia_mean", "iia_std", "patched_accuracy_mean",
                "clean_accuracy_mean", "corrupted_accuracy_mean",
                "recovery_mean", "recovery_std", "logit_flip_mean",
                "logit_flip_std", "sign_flip_rate_mean",
                "sign_flip_rate_std",
            ],
        )

    print("Per-model diagnostic plots:")
    for figure in plot_fixed_layer_run_diagnostics(run_dir):
        print(" -", figure)
        display(Image(filename=str(figure)))
    return run_dir

## 5. GPT-2 Small

In [5]:
GPT2_RUN_DIR = run_and_display("gpt2-small")
print("GPT-2 artifacts:", GPT2_RUN_DIR)

Selectivity models:   0%|          | 0/1 [00:00<?, ?model/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

gpt2-small full_ait validation activations:   0%|          | 0/13 [00:00<?, ?batch/s]

gpt2-small full_ait test activations:   0%|          | 0/35 [00:00<?, ?batch/s]

gpt2-small full_ait tune:   0%|          | 0/1 [00:00<?, ?method/s]

gpt2-small full_ait paired seeds:   0%|          | 0/5 [00:00<?, ?seed/s]

TypeError: clear_device_cache() missing 1 required positional argument: 'device'

## 6. Qwen3-0.6B Base

In [ ]:
QWEN_RUN_DIR = run_and_display("qwen-0.6b")
print("Qwen artifacts:", QWEN_RUN_DIR)

## 7. Aggregate and plot only after every notebook is complete

This cell refuses to aggregate partial studies. Once all eight method/model folders
exist, it writes combined CSVs and the cross-method plots under `RUN_ROOT/combined`.
It is safe to rerun this cell after completing another method notebook.

In [ ]:
try:
    COMBINED_DIR = combine_fixed_layer_selectivity_runs(RUN_ROOT)
except FileNotFoundError as error:
    print(error)
    print("No combined tables or plots were created from this partial study.")
else:
    FIGURES = plot_fixed_layer_selectivity(COMBINED_DIR)
    print("Combined tables:", COMBINED_DIR)
    print("Figures:")
    for figure in FIGURES:
        print(" -", figure)
        display(Image(filename=str(figure)))
    display(pd.read_csv(COMBINED_DIR / "selectivity_summary.csv"))
finally:
    clear_hf_credentials()
    if os.environ.get("HF_TOKEN") is not None:
        raise RuntimeError("HF_TOKEN remains in the notebook environment.")
    if "HF_TOKEN" in _RUNTIME_SECRETS:
        raise RuntimeError("HF_TOKEN remains in the runtime secret cache.")
    print("Verified: HF_TOKEN was cleared from runtime memory and the environment.")